# 15h. Thí nghiệm đổi nhãn: chỉ gán nhãn và đánh giá node có khởi tạo flow (task 5.3 phân tích FPR, GĐ F)

Đề xuất từ phân tích FPR (15a): trên NF-ToN-IoT-v3, phần lớn FP là node chỉ nhận flow. Ở đây huấn luyện lại LightGBM (cấu hình tốt nhất của baseline_search, seed 42) chỉ trên node train có khởi tạo flow và đánh giá trên node val có khởi tạo, so với LightGBM cũ huấn luyện trên mọi node, trên cùng tập node val. **Không đổi dữ liệu, không dựng lại đồ thị**: mặt nạ suy từ đặc trưng `n_out`. CPU, không GPU, không đọc test.

Gắn: output của 04, 11 (đồ thị), 05, 13 (`baseline_search_*.json`). Bật Internet.

In [ ]:
import glob, json, os, resource, subprocess, sys, time

WORK = "/kaggle/working"
REPO = "/tmp/gnn-iot-ids"
INFO = {"device": "CPU"}
T0 = time.time()


def run(cmd, ok=(0,), **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    print(r.stdout[-6000:], end="")
    print(r.stderr[-2000:], end="", file=sys.stderr)
    if r.returncode not in ok:
        raise RuntimeError(f"Lệnh lỗi: {' '.join(cmd)}")
    return r


run(["rm", "-rf", REPO])
run(["git", "clone", "--depth", "1", "https://github.com/minucedev/gnn-iot-ids.git", REPO])
INFO["commit"] = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
with open(f"{REPO}/requirements.txt") as f:
    reqs = [l.strip() for l in f if l.strip() and not l.startswith("#")]
keep = [r for r in reqs if r.split("==")[0].lower() not in {"torch", "jupyter", "ipykernel"}]
run([sys.executable, "-m", "pip", "install", "-q", *keep])
r = run([sys.executable, "-m", "pytest", "-q", "tests/test_fpr.py"], cwd=REPO)
INFO["pytest"] = r.stdout.strip().splitlines()[-1]
print(INFO)

In [ ]:
for ds in ["nf_ton_iot_v3", "nf_bot_iot_v3"]:
    data_dir = os.path.dirname(next(glob.iglob(f"/kaggle/input/**/{ds}/split.json", recursive=True)))
    search = next(glob.iglob(f"/kaggle/input/**/baseline_search_{ds}.json", recursive=True))
    t0 = time.time()
    run([sys.executable, "-I", "scripts/label_pilot.py", data_dir, ds, search, f"{WORK}/label_pilot_{ds}.json"], cwd=REPO)
    INFO[f"seconds_{ds}"] = round(time.time() - t0, 1)
INFO["total_s"] = round(time.time() - T0, 1)
INFO["peak_ram_gb"] = round(max(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
                                resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss) / 2**20, 2)
json.dump(INFO, open(f"{WORK}/run_info.json", "w"), indent=1)
print(INFO)